# 00b FG/RM Foundation Builder

Builds core foundation tables from real source (`Active stock`) for realistic data generation:

- RM master
- FG master (synthetic but anchored to RM families)
- FG-RM BOM mapping
- family-level lead-time priors


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling')
SCRIPTS_DIR = ROOT / 'scripts'
GENERATED_DIR = ROOT / 'outputs' / 'generated'
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

import sys
sys.path.insert(0, str(SCRIPTS_DIR))

from real_source_extraction import extract_active_stock_canonical
from rule_based_synthetic_generator import infer_material_family, demand_speed_bucket

np.random.seed(42)
print('ROOT:', ROOT)

ROOT: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling


In [2]:
canonical = extract_active_stock_canonical().copy()
canonical['rm_family'] = canonical['description'].map(infer_material_family)
canonical['speed_bucket'] = canonical.apply(demand_speed_bucket, axis=1)
canonical['lead_time_days'] = pd.to_numeric(canonical['lead_time_days'], errors='coerce').fillna(30).clip(5, 180)
canonical['future_average'] = pd.to_numeric(canonical['future_average'], errors='coerce').fillna(0).clip(lower=0)
canonical['demand_variance'] = pd.to_numeric(canonical['demand_variance'], errors='coerce').fillna(canonical['future_average'] * 0.2)
canonical[['material_code', 'description', 'rm_family', 'speed_bucket', 'future_average', 'lead_time_days']].head()

,material_code,description,rm_family,speed_bucket,future_average,lead_time_days
0,100005,ETHYL ALCOHOL ( Undenatured ),SOLVENT,CORE,1454.793539,85.0
1,100006,ALCOHOL,SOLVENT,BULK_FAST,2204.317677,85.0
2,100010,ALLANTOIN B.P,ACTIVE,SLOW,41.903800,90.0
3,100012,AMPHISOL,GENERAL,STANDARD,130.043759,60.0
4,100016,F.D.& C BLUE 1,COLORANT,INTERMITTENT,0.538098,60.0


In [3]:
rm_master = canonical[[
    'material_code', 'description', 'rm_family', 'speed_bucket',
    'future_average', 'demand_variance', 'lead_time_days',
    'moq', 'stacking_quantity', 'buffer_stock', 'maximum_stock'
]].copy()
rm_master = rm_master.rename(columns={'material_code': 'rm_code', 'description': 'rm_name'})
rm_master['rm_code'] = rm_master['rm_code'].astype(int)
rm_master = rm_master.sort_values('rm_code').reset_index(drop=True)
rm_master.head()

,rm_code,rm_name,rm_family,speed_bucket,future_average,demand_variance,lead_time_days,moq,stacking_quantity,buffer_stock,maximum_stock
0,100005,ETHYL ALCOHOL ( Undenatured ),SOLVENT,CORE,1454.793539,5981.531992,85.0,1600.0,500.0,268.177459,1868.177459
1,100006,ALCOHOL,SOLVENT,BULK_FAST,2204.317677,523800.505357,85.0,14400.0,640.0,2509.568257,16909.568257
2,100010,ALLANTOIN B.P,ACTIVE,SLOW,41.903800,52.836782,90.0,100.0,75.0,25.935584,125.935584
3,100012,AMPHISOL,GENERAL,STANDARD,130.043759,392.609838,60.0,180.0,400.0,57.724849,237.724849
4,100016,F.D.& C BLUE 1,COLORANT,INTERMITTENT,0.538098,0.018860,60.0,2.0,4.0,0.400082,2.400082


In [4]:
fg_templates = [
    ('SOAP', 'Bar Soap'),
    ('FACEWASH', 'Face Wash'),
    ('SHAMPOO', 'Shampoo'),
    ('CREAM', 'Skin Cream'),
]

n_per_template = 24
fg_rows = []
for code, name in fg_templates:
    for i in range(1, n_per_template + 1):
        fg_rows.append({
            'fg_code': f'{code}_{i:03d}',
            'fg_name': f'{name} {i:03d}',
            'fg_category': code,
        })
fg_master = pd.DataFrame(fg_rows)
fg_master.head()

,fg_code,fg_name,fg_category
0,SOAP_001,Bar Soap 001,SOAP
1,SOAP_002,Bar Soap 002,SOAP
2,SOAP_003,Bar Soap 003,SOAP
3,SOAP_004,Bar Soap 004,SOAP
4,SOAP_005,Bar Soap 005,SOAP


In [5]:
family_pool = {
    'SOAP': ['SURFACTANT', 'OIL_WAX', 'FRAGRANCE_COOLANT', 'COLORANT', 'ACID_BASE'],
    'FACEWASH': ['SURFACTANT', 'ACTIVE', 'FRAGRANCE_COOLANT', 'COLORANT', 'SOLVENT'],
    'SHAMPOO': ['SURFACTANT', 'ACTIVE', 'FRAGRANCE_COOLANT', 'SOLVENT', 'ACID_BASE'],
    'CREAM': ['OIL_WAX', 'ACTIVE', 'FRAGRANCE_COOLANT', 'STARCH_GUM', 'COLORANT'],
}

rng = np.random.default_rng(42)
bom_rows = []
for fg in fg_master.itertuples(index=False):
    fams = family_pool.get(fg.fg_category, ['GENERAL'])
    fam_pick = list(rng.choice(fams, size=min(4, len(fams)), replace=False))
    weights = rng.dirichlet(np.ones(len(fam_pick)) * 1.4)
    for fam, w in zip(fam_pick, weights):
        candidates = rm_master[rm_master['rm_family'] == fam]
        if candidates.empty:
            candidates = rm_master
        rm_row = candidates.sample(n=1, random_state=int(rng.integers(0, 10_000))).iloc[0]
        bom_rows.append({
            'fg_code': fg.fg_code,
            'fg_category': fg.fg_category,
            'rm_code': int(rm_row['rm_code']),
            'rm_family': rm_row['rm_family'],
            'bom_coef': float(np.round(0.15 + 1.6 * w, 4)),
        })

bom = pd.DataFrame(bom_rows)
bom['bom_coef'] = bom.groupby('fg_code')['bom_coef'].transform(lambda s: s / s.sum())
bom.head()

,fg_code,fg_category,rm_code,rm_family,bom_coef
0,SOAP_001,SOAP,100078,ACID_BASE,0.223273
1,SOAP_001,SOAP,100043,SURFACTANT,0.203318
2,SOAP_001,SOAP,101700,COLORANT,0.358864
3,SOAP_001,SOAP,100046,FRAGRANCE_COOLANT,0.214545
4,SOAP_002,SOAP,101415,ACID_BASE,0.300500


In [6]:
lead_priors = (
    rm_master.groupby('rm_family', as_index=False)['lead_time_days']
    .agg(lead_time_p25=lambda s: float(np.percentile(s, 25)),
         lead_time_p50='median',
         lead_time_p75=lambda s: float(np.percentile(s, 75)),
         n_rm='count')
    .sort_values('rm_family')
)
lead_priors

,rm_family,lead_time_p25,lead_time_p50,lead_time_p75,n_rm
0,ACID_BASE,30.0,32.5,65.00,8
1,ACTIVE,60.0,60.0,65.00,9
2,COLORANT,60.0,60.0,63.75,22
3,FRAGRANCE_COOLANT,60.0,60.0,60.00,5
4,GENERAL,60.0,60.0,75.00,205
5,OIL_WAX,60.0,60.0,60.00,22
6,SOLVENT,60.0,60.0,60.00,9
7,STARCH_GUM,47.5,60.0,60.00,3
8,SURFACTANT,60.0,60.0,70.00,5


In [7]:
rm_path = GENERATED_DIR / 'fg_rm_foundation_rm_master.csv'
fg_path = GENERATED_DIR / 'fg_rm_foundation_fg_master.csv'
bom_path = GENERATED_DIR / 'fg_rm_foundation_bom.csv'
lead_path = GENERATED_DIR / 'fg_rm_foundation_leadtime_priors.csv'

rm_master.to_csv(rm_path, index=False)
fg_master.to_csv(fg_path, index=False)
bom.to_csv(bom_path, index=False)
lead_priors.to_csv(lead_path, index=False)

print('Saved:', rm_path)
print('Saved:', fg_path)
print('Saved:', bom_path)
print('Saved:', lead_path)

Saved: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/generated/fg_rm_foundation_rm_master.csv
Saved: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/generated/fg_rm_foundation_fg_master.csv
Saved: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/generated/fg_rm_foundation_bom.csv
Saved: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/generated/fg_rm_foundation_leadtime_priors.csv
